In [57]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    precision_recall_curve,
    classification_report,
)

In [58]:
train_df = pd.read_csv("../data/processed/train_features.csv")
test_df = pd.read_csv("../data/processed/test_features.csv")

# Ensure date is datetime
train_df["date"] = pd.to_datetime(train_df["date"])
test_df["date"] = pd.to_datetime(test_df["date"])

# Quick check
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain columns sample:")
print(train_df.columns[:10].tolist())

Train shape: (1692, 170)
Test shape: (564, 169)

Train columns sample:
['date', '5cm_soil_moist', 'mean_dew_point_temp', 'max_temp', 'sea_level_pressure', 'sst_cameroon_mean_temperature_deg_c', 'sst_cameroon_mean_temperature_uncertainty', 'sst_indian_ocean_mean_temperature_deg_c', 'sst_indian_ocean_mean_temperature_uncertainty', 'potential_water_deficit']


In [59]:
train_df.head(5)

,date,5cm_soil_moist,mean_dew_point_temp,max_temp,sea_level_pressure,sst_cameroon_mean_temperature_deg_c,sst_cameroon_mean_temperature_uncertainty,sst_indian_ocean_mean_temperature_deg_c,sst_indian_ocean_mean_temperature_uncertainty,potential_water_deficit,2m_temp,u10,vapor_pressure_deficit,dryspell_warn_7d,year,month,season_year,5cm_soil_moist_rollmean_7,5cm_soil_moist_rollmin_7,5cm_soil_moist_rollmax_7,5cm_soil_moist_rollmean_14,5cm_soil_moist_rollmin_14,5cm_soil_moist_rollmax_14,5cm_soil_moist_rollmean_30,5cm_soil_moist_rollmin_30,5cm_soil_moist_rollmax_30,mean_dew_point_temp_rollmean_7,mean_dew_point_temp_rollmin_7,mean_dew_point_temp_rollmax_7,mean_dew_point_temp_rollmean_14,mean_dew_point_temp_rollmin_14,mean_dew_point_temp_rollmax_14,mean_dew_point_temp_rollmean_30,mean_dew_point_temp_rollmin_30,mean_dew_point_temp_rollmax_30,max_temp_rollmean_7,max_temp_rollmin_7,max_temp_rollmax_7,max_temp_rollmean_14,max_temp_rollmin_14,max_temp_rollmax_14,max_temp_rollmean_30,max_temp_rollmin_30,max_temp_rollmax_30,sea_level_pressure_rollmean_7,sea_level_pressure_rollmin_7,sea_level_pressure_rollmax_7,sea_level_pressure_rollmean_14,sea_level_pressure_rollmin_14,sea_level_pressure_rollmax_14,sea_level_pressure_rollmean_30,sea_level_pressure_rollmin_30,sea_level_pressure_rollmax_30,sst_cameroon_mean_temperature_deg_c_rollmean_7,sst_cameroon_mean_temperature_deg_c_rollmin_7,sst_cameroon_mean_temperature_deg_c_rollmax_7,sst_cameroon_mean_temperature_deg_c_rollmean_14,sst_cameroon_mean_temperature_deg_c_rollmin_14,sst_cameroon_mean_temperature_deg_c_rollmax_14,sst_cameroon_mean_temperature_deg_c_rollmean_30,sst_cameroon_mean_temperature_deg_c_rollmin_30,sst_cameroon_mean_temperature_deg_c_rollmax_30,sst_cameroon_mean_temperature_uncertainty_rollmean_7,sst_cameroon_mean_temperature_uncertainty_rollmin_7,sst_cameroon_mean_temperature_uncertainty_rollmax_7,sst_cameroon_mean_temperature_uncertainty_rollmean_14,sst_cameroon_mean_temperature_uncertainty_rollmin_14,sst_cameroon_mean_temperature_uncertainty_rollmax_14,sst_cameroon_mean_temperature_uncertainty_rollmean_30,sst_cameroon_mean_temperature_uncertainty_rollmin_30,sst_cameroon_mean_temperature_uncertainty_rollmax_30,sst_indian_ocean_mean_temperature_deg_c_rollmean_7,sst_indian_ocean_mean_temperature_deg_c_rollmin_7,sst_indian_ocean_mean_temperature_deg_c_rollmax_7,sst_indian_ocean_mean_temperature_deg_c_rollmean_14,sst_indian_ocean_mean_temperature_deg_c_rollmin_14,sst_indian_ocean_mean_temperature_deg_c_rollmax_14,sst_indian_ocean_mean_temperature_deg_c_rollmean_30,sst_indian_ocean_mean_temperature_deg_c_rollmin_30,sst_indian_ocean_mean_temperature_deg_c_rollmax_30,sst_indian_ocean_mean_temperature_uncertainty_rollmean_7,sst_indian_ocean_mean_temperature_uncertainty_rollmin_7,sst_indian_ocean_mean_temperature_uncertainty_rollmax_7,sst_indian_ocean_mean_temperature_uncertainty_rollmean_14,sst_indian_ocean_mean_temperature_uncertainty_rollmin_14,sst_indian_ocean_mean_temperature_uncertainty_rollmax_14,sst_indian_ocean_mean_temperature_uncertainty_rollmean_30,sst_indian_ocean_mean_temperature_uncertainty_rollmin_30,sst_indian_ocean_mean_temperature_uncertainty_rollmax_30,potential_water_deficit_rollmean_7,potential_water_deficit_rollmin_7,potential_water_deficit_rollmax_7,potential_water_deficit_rollmean_14,potential_water_deficit_rollmin_14,potential_water_deficit_rollmax_14,potential_water_deficit_rollmean_30,potential_water_deficit_rollmin_30,potential_water_deficit_rollmax_30,2m_temp_rollmean_7,2m_temp_rollmin_7,2m_temp_rollmax_7,2m_temp_rollmean_14,2m_temp_rollmin_14,2m_temp_rollmax_14,2m_temp_rollmean_30,2m_temp_rollmin_30,2m_temp_rollmax_30,u10_rollmean_7,u10_rollmin_7,u10_rollmax_7,u10_rollmean_14,u10_rollmin_14,u10_rollmax_14,u10_rollmean_30,u10_rollmin_30,u10_rollmax_30,vapor_pressure_deficit_rollmean_7,vapor_pressure_deficit_rollmin_7,vapor_pressure_deficit_rollmax_7,vapor_pressure_deficit_rollmean_14,vapor_pressure_deficit_rollmin_14,vapor_pressure_deficit_rollmax_14,vapor_pre

In [60]:
test_df.head()

,date,5cm_soil_moist,mean_dew_point_temp,max_temp,sea_level_pressure,sst_cameroon_mean_temperature_deg_c,sst_cameroon_mean_temperature_uncertainty,sst_indian_ocean_mean_temperature_deg_c,sst_indian_ocean_mean_temperature_uncertainty,potential_water_deficit,2m_temp,u10,vapor_pressure_deficit,year,month,season_year,5cm_soil_moist_rollmean_7,5cm_soil_moist_rollmin_7,5cm_soil_moist_rollmax_7,5cm_soil_moist_rollmean_14,5cm_soil_moist_rollmin_14,5cm_soil_moist_rollmax_14,5cm_soil_moist_rollmean_30,5cm_soil_moist_rollmin_30,5cm_soil_moist_rollmax_30,mean_dew_point_temp_rollmean_7,mean_dew_point_temp_rollmin_7,mean_dew_point_temp_rollmax_7,mean_dew_point_temp_rollmean_14,mean_dew_point_temp_rollmin_14,mean_dew_point_temp_rollmax_14,mean_dew_point_temp_rollmean_30,mean_dew_point_temp_rollmin_30,mean_dew_point_temp_rollmax_30,max_temp_rollmean_7,max_temp_rollmin_7,max_temp_rollmax_7,max_temp_rollmean_14,max_temp_rollmin_14,max_temp_rollmax_14,max_temp_rollmean_30,max_temp_rollmin_30,max_temp_rollmax_30,sea_level_pressure_rollmean_7,sea_level_pressure_rollmin_7,sea_level_pressure_rollmax_7,sea_level_pressure_rollmean_14,sea_level_pressure_rollmin_14,sea_level_pressure_rollmax_14,sea_level_pressure_rollmean_30,sea_level_pressure_rollmin_30,sea_level_pressure_rollmax_30,sst_cameroon_mean_temperature_deg_c_rollmean_7,sst_cameroon_mean_temperature_deg_c_rollmin_7,sst_cameroon_mean_temperature_deg_c_rollmax_7,sst_cameroon_mean_temperature_deg_c_rollmean_14,sst_cameroon_mean_temperature_deg_c_rollmin_14,sst_cameroon_mean_temperature_deg_c_rollmax_14,sst_cameroon_mean_temperature_deg_c_rollmean_30,sst_cameroon_mean_temperature_deg_c_rollmin_30,sst_cameroon_mean_temperature_deg_c_rollmax_30,sst_cameroon_mean_temperature_uncertainty_rollmean_7,sst_cameroon_mean_temperature_uncertainty_rollmin_7,sst_cameroon_mean_temperature_uncertainty_rollmax_7,sst_cameroon_mean_temperature_uncertainty_rollmean_14,sst_cameroon_mean_temperature_uncertainty_rollmin_14,sst_cameroon_mean_temperature_uncertainty_rollmax_14,sst_cameroon_mean_temperature_uncertainty_rollmean_30,sst_cameroon_mean_temperature_uncertainty_rollmin_30,sst_cameroon_mean_temperature_uncertainty_rollmax_30,sst_indian_ocean_mean_temperature_deg_c_rollmean_7,sst_indian_ocean_mean_temperature_deg_c_rollmin_7,sst_indian_ocean_mean_temperature_deg_c_rollmax_7,sst_indian_ocean_mean_temperature_deg_c_rollmean_14,sst_indian_ocean_mean_temperature_deg_c_rollmin_14,sst_indian_ocean_mean_temperature_deg_c_rollmax_14,sst_indian_ocean_mean_temperature_deg_c_rollmean_30,sst_indian_ocean_mean_temperature_deg_c_rollmin_30,sst_indian_ocean_mean_temperature_deg_c_rollmax_30,sst_indian_ocean_mean_temperature_uncertainty_rollmean_7,sst_indian_ocean_mean_temperature_uncertainty_rollmin_7,sst_indian_ocean_mean_temperature_uncertainty_rollmax_7,sst_indian_ocean_mean_temperature_uncertainty_rollmean_14,sst_indian_ocean_mean_temperature_uncertainty_rollmin_14,sst_indian_ocean_mean_temperature_uncertainty_rollmax_14,sst_indian_ocean_mean_temperature_uncertainty_rollmean_30,sst_indian_ocean_mean_temperature_uncertainty_rollmin_30,sst_indian_ocean_mean_temperature_uncertainty_rollmax_30,potential_water_deficit_rollmean_7,potential_water_deficit_rollmin_7,potential_water_deficit_rollmax_7,potential_water_deficit_rollmean_14,potential_water_deficit_rollmin_14,potential_water_deficit_rollmax_14,potential_water_deficit_rollmean_30,potential_water_deficit_rollmin_30,potential_water_deficit_rollmax_30,2m_temp_rollmean_7,2m_temp_rollmin_7,2m_temp_rollmax_7,2m_temp_rollmean_14,2m_temp_rollmin_14,2m_temp_rollmax_14,2m_temp_rollmean_30,2m_temp_rollmin_30,2m_temp_rollmax_30,u10_rollmean_7,u10_rollmin_7,u10_rollmax_7,u10_rollmean_14,u10_rollmin_14,u10_rollmax_14,u10_rollmean_30,u10_rollmin_30,u10_rollmax_30,vapor_pressure_deficit_rollmean_7,vapor_pressure_deficit_rollmin_7,vapor_pressure_deficit_rollmax_7,vapor_pressure_deficit_rollmean_14,vapor_pressure_deficit_rollmin_14,vapor_pressure_deficit_rollmax_14,vapor_pressure_deficit_rol

In [61]:
target_col = "dryspell_warn_7d"

drop_cols = ["date", "season_year", target_col]

if "year" in train_df.columns:
    drop_cols.append("year")

feature_cols = [col for col in train_df.columns if col not in drop_cols]
print("Number of features:", len(feature_cols))
print(feature_cols[:20])

Number of features: 166
['5cm_soil_moist', 'mean_dew_point_temp', 'max_temp', 'sea_level_pressure', 'sst_cameroon_mean_temperature_deg_c', 'sst_cameroon_mean_temperature_uncertainty', 'sst_indian_ocean_mean_temperature_deg_c', 'sst_indian_ocean_mean_temperature_uncertainty', 'potential_water_deficit', '2m_temp', 'u10', 'vapor_pressure_deficit', 'month', '5cm_soil_moist_rollmean_7', '5cm_soil_moist_rollmin_7', '5cm_soil_moist_rollmax_7', '5cm_soil_moist_rollmean_14', '5cm_soil_moist_rollmin_14', '5cm_soil_moist_rollmax_14', '5cm_soil_moist_rollmean_30']


In [62]:
train_mask =train_df["season_year"]<= 2016
val_mask =train_df["season_year"]>= 2017

train_part = train_df.loc[train_mask].copy()
val_part = train_df.loc[val_mask].copy()

X_train = train_part[feature_cols]
y_train = train_part[target_col]

X_val = val_part[feature_cols]
y_val = val_part[target_col]

print("Train seasons:", sorted(train_part["season_year"].unique()))
print("Validation seasons:", sorted(val_part["season_year"].unique()))
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

Train seasons: [np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016)]
Validation seasons: [np.int64(2017), np.int64(2018), np.int64(2019)]
X_train shape: (1410, 166)
X_val shape: (282, 166)


In [65]:
models = {
    "logreg":Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        ))
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "hist_gb": HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=6,
        max_iter= 300,
        random_state=42
    )
}

In [66]:
def evaluate_model(name, model, X_train, y_train, X_val, y_val):
    # Fit
    if name == "hist_gb":
        # Approx balanced sample weights
        class_counts = y_train.value_counts()
        w0 = 1.0
        w1 = class_counts[0] / class_counts[1]
        sample_weight = y_train.map({0: w0, 1: w1})
        model.fit(X_train, y_train, sample_weight=sample_weight)
    else:
        model.fit(X_train, y_train)
    # Predict proba
    if hasattr(model, "predict_proba"):
        val_probs = model.predict_proba(X_val)[:, 1]
    else:
        # fallback for models without predict_proba
        val_probs = model.decision_function(X_val)
    # Find best threshold by F1
    precision, recall, thresholds = precision_recall_curve(y_val, val_probs)
    f1_scores = (2 * precision * recall) / (precision + recall + 1e-9)
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    val_pred = (val_probs >= best_threshold).astype(int)
    metrics = {
        "model": name,
        "best_threshold": best_threshold,
        "f1": f1_score(y_val, val_pred),
        "precision": precision_score(y_val, val_pred),
        "recall": recall_score(y_val, val_pred),
        "confusion": confusion_matrix(y_val, val_pred)
    }
    return metrics, val_probs

In [68]:
results = []
for name, model in models.items():
    metrics, _ = evaluate_model(name, model, X_train, y_train, X_val, y_val)
    results.append(metrics)
results_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != "confusion"} for r in results
]).sort_values("f1", ascending=False)
print(results_df)
for r in results:
    print(f"\n{r['model']} confusion matrix:")
    print(r["confusion"])

           model  best_threshold    f1  precision  recall
1  random_forest            0.15  0.48       0.32    1.00
2        hist_gb            0.02  0.41       0.27    0.86
0         logreg            0.02  0.34       0.22    0.71

logreg confusion matrix:
[[185  69]
 [  8  20]]

random_forest confusion matrix:
[[194  60]
 [  0  28]]

hist_gb confusion matrix:
[[189  65]
 [  4  24]]


In [71]:
# Refit best model on full training data
best_threshold = 0.15

X_full = train_df[feature_cols]
y_full = train_df[target_col]
best_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
best_model.fit(X_full, y_full)
# Predict on test
X_test = test_df[feature_cols]
test_probs = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_probs >= best_threshold).astype(int)
pred_df = pd.DataFrame({
    "date": test_df["date"],
    "prob_dryspell": test_probs,
    "pred_label": test_pred
})
pred_df.to_csv("../data/processed/test_predictions.csv", index=False)
pred_df.head()

,date,prob_dryspell,pred_label
0,2020-07-30,5.39e-03,0
1,2020-07-31,5.39e-03,0
2,2020-08-01,2.70e-03,0
3,2020-08-02,4.64e-03,0
4,2020-08-03,6.03e-03,0


In [72]:
pred_df["pred_label"].mean()
pred_df["prob_dryspell"].describe() 

count    564.00
mean       0.08
std        0.09
min        0.00
25%        0.02
50%        0.05
75%        0.13
max        0.57
Name: prob_dryspell, dtype: float64